In [2]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# Localiza a pasta raiz que contém 'src' e 'dados'
candidatos = [Path(".").resolve(), Path("..").resolve(), Path("../..").resolve()]
raiz = next(
    (p for p in candidatos if (p / "src").exists() and (p / "dados").exists()),
    Path(".").resolve(),
)

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.preprocessamento import (
    carregar_dados,
    preparar_dados,
    criar_preprocessador,
    criar_particoes,
)
from src.avaliacao import avaliar_modelo

caminho_csv = raiz / "dados" / "online_shoppers_convertido.csv"
df = carregar_dados(caminho_csv)
X, y = preparar_dados(df)
particoes = criar_particoes(X, y)

print("Raiz do projeto:", raiz)
print("Shape de X:", X.shape)
print("Distribuição de y:\n", pd.Series(y).value_counts(dropna=False))

Raiz do projeto: C:\Users\Jose Augusto\Documents\GitHub\Trabalho-2-INE
Shape de X: (12205, 17)
Distribuição de y:
 Revenue
0    10297
1     1908
Name: count, dtype: int64


In [ ]:
# Configuração 1: Modelo base equilibrado
modelo_xgb_1 = Pipeline(
    steps=[
        ("preprocessador", criar_preprocessador()),
        (
            "classificador",
            XGBClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="logloss",
                n_jobs=-1,
            ),
        ),
    ]
)

# Configuração 2: Maior capacidade de ajuste com taxa de aprendizado menor e regularização
modelo_xgb_2 = Pipeline(
    steps=[
        ("preprocessador", criar_preprocessador()),
        (
            "classificador",
            XGBClassifier(
                n_estimators=400,
                learning_rate=0.03,
                max_depth=6,
                min_child_weight=3,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="logloss",
                n_jobs=-1,
            ),
        ),
    ]
)

metricas_1, matriz_1 = avaliar_modelo(modelo_xgb_1, X, y, particoes)
metricas_2, matriz_2 = avaliar_modelo(modelo_xgb_2, X, y, particoes)

print("=== Configuração 1 — Métricas por fold ===")
display(metricas_1)
print("=== Configuração 1 — Média das métricas ===")
display(metricas_1.mean(numeric_only=True))
print("Matriz de confusão agregada (Configuração 1):")
print(matriz_1)

print("\n=== Configuração 2 — Métricas por fold ===")
display(metricas_2)
print("=== Configuração 2 — Média das métricas ===")
display(metricas_2.mean(numeric_only=True))
print("Matriz de confusão agregada (Configuração 2):")
print(matriz_2)

=== Configuração 1 — Métricas por fold ===


,Acurácia,Precisão,Recall,F1-score
Fold,,,,
1,0.907415,0.744479,0.619423,0.676218
2,0.906596,0.754153,0.595801,0.665689
3,0.901680,0.721875,0.604712,0.658120
4,0.900451,0.709970,0.615183,0.659187
5,0.903728,0.736334,0.599476,0.660895


=== Configuração 1 — Média das métricas ===


Acurácia    0.903974
Precisão    0.733362
Recall      0.606919
F1-score    0.664022
dtype: float64

Matriz de confusão agregada (Configuração 1):
[[9875  422]
 [ 750 1158]]

=== Configuração 2 — Métricas por fold ===


,Acurácia,Precisão,Recall,F1-score
Fold,,,,
1,0.899631,0.712500,0.598425,0.650499
2,0.903318,0.750865,0.569554,0.647761
3,0.902909,0.731629,0.599476,0.658993
4,0.900041,0.705357,0.620419,0.660167
5,0.902089,0.728435,0.596859,0.656115


=== Configuração 2 — Média das métricas ===


Acurácia    0.901598
Precisão    0.725757
Recall      0.596947
F1-score    0.654707
dtype: float64

Matriz de confusão agregada (Configuração 2):
[[9865  432]
 [ 769 1139]]


In [4]:
import pandas as pd

resumo_xgb = pd.DataFrame(
    {
        "Config 1 (lr=0.05, depth=4)": metricas_1.mean(numeric_only=True),
        "Config 2 (lr=0.03, depth=6)": metricas_2.mean(numeric_only=True),
    }
).T

display(resumo_xgb)

,Acurácia,Precisão,Recall,F1-score
"Config 1 (lr=0.05, depth=4)",0.903974,0.733362,0.606919,0.664022
"Config 2 (lr=0.03, depth=6)",0.901598,0.725757,0.596947,0.654707


## Comparação das métricas

As barras representam as médias da validação cruzada estratificada de cinco folds. A classe positiva é a sessão com compra. Os arquivos gerados por este notebook usam o prefixo `xgboost_script_` em `resultados/`.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

saida = raiz / "resultados"
saida.mkdir(exist_ok=True)

metricas_por_fold = pd.concat(
    [metricas_1.assign(configuracao="Configuração 1"),
     metricas_2.assign(configuracao="Configuração 2")],
    names=["Fold"],
).reset_index()
metricas_por_fold.to_csv(saida / "xgboost_script_metricas_por_fold.csv", index=False, encoding="utf-8-sig")
resumo_xgb.to_csv(saida / "xgboost_script_metricas_resumo.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(9, 4.5))
resumo_xgb.plot.bar(ax=ax, rot=0)
ax.set_title("XGBoost: métricas médias por configuração")
ax.set_ylabel("Média nos cinco folds")
ax.set_ylim(0, 1)
ax.legend(title="Métrica", loc="lower right")
plt.tight_layout()
plt.savefig(saida / "xgboost_script_metricas.png", dpi=300)
plt.show()
plt.close(fig)


## Matrizes de confusão

Cada matriz reúne previsões fora da amostra dos cinco folds. As linhas indicam a classe real e as colunas a classe prevista; a classe positiva é compra. A célula também salva cada matriz em CSV.


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(10, 4))
for eixo, nome, matriz, numero in zip(
    eixos, ["Configuração 1", "Configuração 2"], [matriz_1, matriz_2], [1, 2]
):
    ConfusionMatrixDisplay(matriz, display_labels=["Sem compra", "Compra"]).plot(
        ax=eixo, cmap="Blues", colorbar=False, values_format="d"
    )
    eixo.set_title(nome)
    eixo.set_xlabel("Previsto")
    eixo.set_ylabel("Real")
    pd.DataFrame(
        matriz,
        index=["real_sem_compra", "real_compra"],
        columns=["previsto_sem_compra", "previsto_compra"],
    ).to_csv(saida / f"xgboost_script_matriz_{numero}.csv", encoding="utf-8-sig")
fig.tight_layout()
fig.savefig(saida / "xgboost_script_matrizes_confusao.png", dpi=300)
plt.show()
plt.close(fig)


## Importância das variáveis

A configuração com maior F1 médio é novamente ajustada em todos os dados apenas para descrever as dez variáveis transformadas mais importantes por ganho. As métricas e matrizes acima continuam sendo as da validação cruzada. Importância não demonstra causalidade.


In [ ]:
from sklearn.base import clone

modelos = {"Config 1 (lr=0.05, depth=4)": modelo_xgb_1,
           "Config 2 (lr=0.03, depth=6)": modelo_xgb_2}
nome_final = resumo_xgb["F1-score"].idxmax()
modelo_final = clone(modelos[nome_final]).fit(X, y)
nomes = modelo_final.named_steps["preprocessador"].get_feature_names_out()
importancias = pd.Series(
    modelo_final.named_steps["classificador"].feature_importances_,
    index=nomes,
    name="ganho",
).sort_values(ascending=False)
print("Configuração com maior F1 médio:", nome_final)
display(importancias.head(10).round(4).to_frame())
importancias.to_csv(saida / "xgboost_script_importancias.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8, 4.5))
importancias.head(10).iloc[::-1].plot.barh(ax=ax)
ax.set_title("Importância das variáveis — " + nome_final)
ax.set_xlabel("Ganho relativo")
plt.tight_layout()
plt.savefig(saida / "xgboost_script_importancia_variaveis.png", dpi=300)
plt.show()
plt.close(fig)
